In [21]:
!pip install transformers --upgrade



  Obtaining dependency information for transformers from https://files.pythonhosted.org/packages/96/f2/25b27b396af03d5b64e61976b14f7209e2939e9e806c10749b6d277c273e/transformers-4.52.4-py3-none-any.whl.metadata
  Using cached transformers-4.52.4-py3-none-any.whl.metadata (38 kB)
Using cached transformers-4.52.4-py3-none-any.whl (10.5 MB)


In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from datasets import Dataset
import json


In [9]:
# Load dữ liệu từ file JSON
with open("data/train_dataset_test.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

dataset = Dataset.from_dict({
    "text": [item["text"] for item in raw_data],
    "summary": [item["summary"] for item in raw_data],
})


In [10]:
# Load tokenizer và model từ VietAI
model_name = "VietAI/vit5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [13]:
# Hàm tiền xử lý dữ liệu
def preprocess_function(examples):
    inputs = tokenizer(examples["text"], max_length=512, truncation=True, padding="max_length")
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["summary"], max_length=150, truncation=True, padding="max_length")
    inputs["labels"] = labels["input_ids"]
    return inputs

tokenized_data = dataset.map(preprocess_function, batched=True)


Map:   0%|          | 0/55 [00:00<?, ? examples/s]

In [15]:

# Cấu hình huấn luyện

training_args = Seq2SeqTrainingArguments(
    output_dir="./text_summarization_model",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    save_steps=100,
    save_total_limit=1,
)


In [16]:
# Tạo Trainer và huấn luyện mô hình
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model),
)

trainer.train()
trainer.save_model("./text_summarization_model")


C:\Users\admin\AppData\Local\Temp\ipykernel_15584\1142748840.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss
10,2.513600
20,1.689000
30,1.659300
40,1.408500
50,1.295600
60,1.174300
70,1.086800
80,0.982600
90,1.031700
100,0.917500


c:\Users\admin\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\admin\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [9]:

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import Dataset
from sklearn.metrics import classification_report
from rouge_score import rouge_scorer
import json


In [10]:
model_path = "./text_summarization_model_1"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)


In [11]:
with open("data_testing/test_dataset_2.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

test_dataset = Dataset.from_dict({
    "text": [item["text"] for item in raw_data],
    "summary": [item["summary"] for item in raw_data],
})


In [12]:
def generate_summary(example):
    inputs = tokenizer(example["text"], return_tensors="pt", max_length=512, truncation=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    summary_ids = model.generate(**inputs, max_length=150, num_beams=4)
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


In [16]:
print("🔄 Đang sinh tóm tắt cho tập test...")
generated_summaries = []

for i, item in enumerate(test_dataset):
    summary = generate_summary(item)
    generated_summaries.append(summary)
    print(f"✅ Đã sinh tóm tắt cho một bài viết. {i+1}/{len(test_dataset)}")


🔄 Đang sinh tóm tắt cho tập test...
✅ Đã sinh tóm tắt cho một bài viết. 1/77
✅ Đã sinh tóm tắt cho một bài viết. 2/77
✅ Đã sinh tóm tắt cho một bài viết. 3/77
✅ Đã sinh tóm tắt cho một bài viết. 4/77
✅ Đã sinh tóm tắt cho một bài viết. 5/77
✅ Đã sinh tóm tắt cho một bài viết. 6/77
✅ Đã sinh tóm tắt cho một bài viết. 7/77
✅ Đã sinh tóm tắt cho một bài viết. 8/77
✅ Đã sinh tóm tắt cho một bài viết. 9/77
✅ Đã sinh tóm tắt cho một bài viết. 10/77
✅ Đã sinh tóm tắt cho một bài viết. 11/77
✅ Đã sinh tóm tắt cho một bài viết. 12/77
✅ Đã sinh tóm tắt cho một bài viết. 13/77
✅ Đã sinh tóm tắt cho một bài viết. 14/77
✅ Đã sinh tóm tắt cho một bài viết. 15/77
✅ Đã sinh tóm tắt cho một bài viết. 16/77
✅ Đã sinh tóm tắt cho một bài viết. 17/77
✅ Đã sinh tóm tắt cho một bài viết. 18/77
✅ Đã sinh tóm tắt cho một bài viết. 19/77
✅ Đã sinh tóm tắt cho một bài viết. 20/77
✅ Đã sinh tóm tắt cho một bài viết. 21/77
✅ Đã sinh tóm tắt cho một bài viết. 22/77
✅ Đã sinh tóm tắt cho một bài viết. 23/77
✅ Đã si

In [25]:
# Tạo đối tượng ROUGE scorer
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

# Tạo danh sách lưu kết quả
rouge1_p, rouge1_r, rouge1_f = [], [], []
rouge2_p, rouge2_r, rouge2_f = [], [], []
rougeL_p, rougeL_r, rougeL_f = [], [], []

# Tính ROUGE cho từng mẫu
for pred, ref in zip(generated_summaries, test_dataset["summary"]):
    scores = scorer.score(ref, pred)
    rouge1_p.append(scores["rouge1"].precision)
    rouge1_r.append(scores["rouge1"].recall)
    rouge1_f.append(scores["rouge1"].fmeasure)

    rouge2_p.append(scores["rouge2"].precision)
    rouge2_r.append(scores["rouge2"].recall)
    rouge2_f.append(scores["rouge2"].fmeasure)

    rougeL_p.append(scores["rougeL"].precision)
    rougeL_r.append(scores["rougeL"].recall)
    rougeL_f.append(scores["rougeL"].fmeasure)


In [26]:
# Hàm in ra kết quả đẹp
def print_rouge_score(name, p_list, r_list, f_list):
    print(f"\n== {name.upper()} ==")
    print(f"Precision: {sum(p_list)/len(p_list):.4f}")
    print(f"Recall:    {sum(r_list)/len(r_list):.4f}")
    print(f"F1-Score:  {sum(f_list)/len(f_list):.4f}")

# In kết quả
print("📊 KẾT QUẢ ĐÁNH GIÁ ROUGE:")
print_rouge_score("ROUGE-1", rouge1_p, rouge1_r, rouge1_f)
print_rouge_score("ROUGE-2", rouge2_p, rouge2_r, rouge2_f)
print_rouge_score("ROUGE-L", rougeL_p, rougeL_r, rougeL_f)


📊 KẾT QUẢ ĐÁNH GIÁ ROUGE:

== ROUGE-1 ==
Precision: 0.8241
Recall:    0.8267
F1-Score:  0.8174

== ROUGE-2 ==
Precision: 0.6557
Recall:    0.6564
F1-Score:  0.6499

== ROUGE-L ==
Precision: 0.6739
Recall:    0.6749
F1-Score:  0.6681


In [ ]:
# Bắt đầu test với một bài viết cụ thể

In [28]:
model_path = "./text_summarization_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)


In [29]:
def summarize_text(text, max_input_length=512, max_output_length=150):
    inputs = tokenizer(text, return_tensors="pt", max_length=max_input_length, truncation=True)
    summary_ids = model.generate(inputs["input_ids"], max_length=max_output_length, num_beams=4, early_stopping=True)
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


In [30]:
sample_text = """
Sáng 27/5/2025, Hiệu trưởng Trường Đại học Kiến trúc Hà Nội (HAU) - PGS.TS.KTS. Lê Quân đã có buổi tiếp và làm việc với đoàn công tác đến từ Trường Đại học Swinburne Australia do GS. Blair Kuys - Hiệu trưởng, Trường Kiến trúc và Thiết kế dẫn đầu.
Cùng đi trong đoàn Trường Đại học Swinburne có GS. Emad Gad - Hiệu trưởng, Trường Kỹ thuật; TS. Phương Quốc Định - Giám đốc Chương trình Kiến trúc Nội thất; TS. Chris Lewis - Giám đốc Điều hành, Bộ phận Sinh viên và quảng bá và cô Trần Hà - Trưởng đại diện Tổ chức giáo dục TET tại Việt Nam.
Tiếp đoàn, về phía Trường Đại học Kiến trúc Hà Nội có PGS.TS. Nguyễn Ngọc Phương - Trưởng Khoa Đào tạo Sau đại học; PGS.TS. Đặng Vũ Hiệp - Phó Trưởng Khoa Xây dựng và TS. Nguyễn Minh Nhất - Giám đốc Trung tâm Hợp tác quốc tế, Viện Đào tạo và Hợp tác quốc tế.
Thay mặt Lãnh đạo Nhà trường, PGS.TS.KTS. Lê Quân bày tỏ vui mừng được đón tiếp các Giáo sư Trường Đại học Swinburne sang thăm và làm việc tại HAU. Hiệu trưởng Lê Quân đánh giá cao sự gắn bó và hợp tác chặt chẽ giữa HAU với các Trường Đại học Australia nói chung và Trường Đại học Swinburne nói riêng trong thời gian qua và cho rằng sự gặp mặt lần này có tính chất “làm mới” Thỏa thuận hợp tác đã được ký kết trước đây giữa hai Trường.
Tại buổi làm việc, lãnh đạo hai bên đã cùng nhau trao đổi các cơ hội hợp tác về đào tạo, xây dựng các chương trình chuyển tiếp; hợp tác trong chương trình nghiên cứu sinh, học bổng bổ sung cho Đề án 89 và các học bổng nghiên cứu khác; hợp tác trong nghiên cứu khoa học, trao đổi giảng viên, sinh viên các chuyên ngành Kiến trúc, Nội thất, tổ chức các hội thảo, workshop…
Đại diện Swinburne cảm ơn sự đón tiếp nồng nhiệt của lãnh đạo HAU đối với đoàn và khẳng định đây là cơ hội để hai Trường tăng thêm sự hợp tác, giao lưu, học hỏi lẫn nhau. Phía Swinburne cũng tin tưởng rằng với các thỏa thuận được ký kết với Đại học Kiến trúc Hà Nội trước đây và hiện tại, trên tinh thần quan hệ hữu nghị và hợp tác truyền thống, hai bên sẽ có các buổi làm việc tiếp theo nhằm hiện thực hóa các nội dung của các chương trình liên kết.
Buổi làm việc giữa lãnh đạo hai Trường thể hiện những định hướng cùng với chiến lược đào tạo. Lãnh đạo HAU ghi nhận các ý kiến trao đổi, hoan nghênh việc hợp tác cùng đại diện lãnh đạo Swinburne, đồng thời bày tỏ mong muốn trong thời gian tới sẽ mở ra mối quan hệ hợp tác mới, hướng tới xây dựng những chương trình hợp tác về mọi mặt.
Trường Đại học Swinburne là Trường Đại học danh tiếng tại Úc và trên thế giới. Trường được thành lập từ năm 1908 theo tên của kỹ sư, chính trị gia người Úc George Swinburne đặt tại miền Đông của thành phố Melbourne. Với lịch sử hơn 100 năm trong lĩnh vực giáo dục, đào tạo Swinburne luôn nằm trong top 400 Trường Đại học tốt nhất thế giới.
"""

summary = summarize_text(sample_text)
print("Tóm tắt:", summary)


Tóm tắt: PGS.TS.KTS. Lê Quân, Hiệu trưởng Trường Đại học Kiến trúc Hà Nội, tiếp và làm việc với đoàn công tác từ Trường Đại học Swinburne Australia. Hai bên thảo luận về việc tăng cường hợp tác trong đào tạo, nghiên cứu, trao đổi giảng viên và sinh viên, tổ chức hội thảo, workshop. Swinburne đánh giá cao tinh thần hợp tác của HAU và hy vọng mối quan hệ giữa hai trường sẽ ngày càng phát triển. Swinburne hy vọng mối quan hệ giữa hai trường sẽ ngày càng phát triển. Swinburne hy vọng mối quan hệ giữa hai trường sẽ ngày càng phát triển và mở ra nhiều cơ hội hợp tác mới giữa hai Trường.
